In [1]:
import pandas as pd
import os 
os.getcwd()

'c:\\Users\\hamoud\\Desktop\\reddit_jokes_project\\data'

In [3]:
data = pd.read_csv("./raw/data.csv")
data.head(5)

,type,id,subreddit.id,subreddit.name,subreddit.nsfw,created_utc,permalink,domain,url,selftext,title,score
0,post,ftbp1i,2qh72,jokes,False,1585785543,https://old.reddit.com/r/Jokes/comments/ftbp1i...,self.jokes,NaN,My corona is covered with foreskin so it is no...,I am soooo glad I'm not circumcised!,2
1,post,ftboup,2qh72,jokes,False,1585785522,https://old.reddit.com/r/Jokes/comments/ftboup...,self.jokes,NaN,It's called Google Sheets.,Did you know Google now has a platform for rec...,9
2,post,ftbopj,2qh72,jokes,False,1585785508,https://old.reddit.com/r/Jokes/comments/ftbopj...,self.jokes,NaN,The vacuum doesn't snore after sex.\n\n&amp;#x...,What is the difference between my wife and my ...,15
3,post,ftbnxh,2qh72,jokes,False,1585785428,https://old.reddit.com/r/Jokes/comments/ftbnxh...,self.jokes,NaN,[removed],My last joke for now.,9
4,post,ftbjpg,2qh72,jokes,False,1585785009,https://old.reddit.com/r/Jokes/comments/ftbjpg...,self.jokes,NaN,[removed],The Nintendo 64 turns 18 this week...,134


In [ ]:

import os, re, json
import pandas as pd
from sklearn.model_selection import train_test_split

RAW_PATH = r"./raw/data.csv"
OUT_DIR = "./processed"
SPLIT_DIR = "./splits"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)


LEVELS = ["easy", "medium", "hard"]
SETTINGS = ["office", "school", "family dinner", "online", "bar"]
TONES = ["sarcastic", "dry", "wholesome", "absurd"]


SAFE_MODE = True

BAD_PATTERNS = [
    r"\brape\b",
    r"\bsuicide\b",
    r"\bkill yourself\b",
    r"\bnigger\b",
    r"\bfaggot\b",
    r"\bcunt\b",
    r"\bincest\b",
    r"\bchild porn\b",
]

PROFANITY = [r"\bfuck\b", r"\bshit\b", r"\bcock\b", r"\bpussy\b", r"\bdick\b"]

def normalize_text(s: str) -> str:
    s = str(s)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"&amp;#x200B;|&amp;#x...|&amp;#x[0-9a-fA-F]+;", " ", s)  
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def infer_setting(text: str) -> str:
    t = text.lower()
    if any(w in t for w in ["boss", "meeting", "email", "deadline", "office", "coworker", "slack"]):
        return "office"
    if any(w in t for w in ["teacher", "class", "school", "homework", "exam", "student"]):
        return "school"
    if any(w in t for w in ["mom", "dad", "grandma", "dinner", "family", "thanksgiving"]):
        return "family dinner"
    if any(w in t for w in ["reddit", "twitter", "online", "wifi", "google", "internet"]):
        return "online"
    if any(w in t for w in ["beer", "bartender", "bar", "whiskey", "pub"]):
        return "bar"
    
    return "online"

def infer_tone(text: str) -> str:
    t = text.lower()
    if any(w in t for w in ["wholesome", "thank you", "love", "kind", "sweet", "hug"]):
        return "wholesome"
    if any(w in t for w in ["yeah right", "as if", "sure", "obviously", "great..."]):
        return "sarcastic"
    if any(w in t for w in ["nonsense", "random", "banana", "alien", "absurd", "weird"]):
        return "absurd"
   
    return "dry"

def infer_level(text: str) -> str:
    
    L = len(text)
    n_sent = len(re.findall(r"[.!?]", text))
    if L < 120 and n_sent <= 2:
        return "easy"
    if L < 250 and n_sent <= 3:
        return "medium"
    return "hard"

def make_instruction(level: str, setting: str, tone: str) -> str:
    return (
        "Write a joke.\n"
        f"LEVEL: {level}\n"
        f"SETTING: {setting}\n"
        f"TONE: {tone}"
    )

def contains_any_pattern(text: str, patterns) -> bool:
    for pat in patterns:
        if re.search(pat, text, flags=re.IGNORECASE):
            return True
    return False

def main():
    print("Loading:", RAW_PATH)
    df = pd.read_csv(RAW_PATH)
    print("Initial shape:", df.shape)
    print("Columns:", df.columns.tolist())

    df = df[df["url"].isna()]

    df = df[~df["selftext"].isin(["[deleted]", "[removed]"])]

    df = df.dropna(subset=["title"])

    df["selftext"] = df["selftext"].fillna("").astype(str)
    df["title"] = df["title"].fillna("").astype(str)

    df["joke"] = (df["title"].str.strip() + "\n" + df["selftext"].str.strip()).str.strip()
    df["joke"] = df["joke"].map(normalize_text)

    
    df = df[df["joke"].str.len() > 0]

    df["joke"] = df["joke"].str.replace(r"(?is)\n*edit:.*$", "", regex=True)
    df["joke"] = df["joke"].str.replace(r"_{5,}", " ", regex=True)

    df = df[df["joke"].str.len() >= 30]
    df = df[df["joke"].str.len() <= 400]  
    df = df[df["joke"].str.count(r"[.!?]") <= 3]

 
    if SAFE_MODE:
        df = df[~df["joke"].map(lambda t: contains_any_pattern(t, BAD_PATTERNS))]
        
        df = df[~df["joke"].map(lambda t: contains_any_pattern(t, PROFANITY))]

  
    df = df.drop_duplicates(subset=["joke"]).reset_index(drop=True)

    print("After cleaning:", df.shape)

    
    df["LEVEL"] = df["joke"].map(infer_level)
    df["SETTING"] = df["joke"].map(infer_setting)
    df["TONE"] = df["joke"].map(infer_tone)

    df["instruction"] = df.apply(lambda r: make_instruction(r["LEVEL"], r["SETTING"], r["TONE"]), axis=1)
    df["response"] = df["joke"]

    final = df[["instruction", "response"]].reset_index(drop=True)

    train, temp = train_test_split(final, test_size=0.2, random_state=42)
    val, test = train_test_split(temp, test_size=0.5, random_state=42)

    final.to_csv(os.path.join(OUT_DIR, "all_clean.csv"), index=False)

    def write_jsonl(path, frame):
        with open(path, "w", encoding="utf-8") as f:
            for _, row in frame.iterrows():
                f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

    write_jsonl(os.path.join(SPLIT_DIR, "train.jsonl"), train)
    write_jsonl(os.path.join(SPLIT_DIR, "val.jsonl"), val)
    write_jsonl(os.path.join(SPLIT_DIR, "test.jsonl"), test)

    print("Saved:")
    print(" train:", len(train))
    print(" val:", len(val))
    print(" test:", len(test))
    print("Example instruction:\n", train.iloc[0]["instruction"])
    print("Example response:\n", train.iloc[0]["response"][:200], "...")

if __name__ == "__main__":
    main()



Loading: ./raw/data.csv
Initial shape: (1000000, 12)
Columns: ['type', 'id', 'subreddit.id', 'subreddit.name', 'subreddit.nsfw', 'created_utc', 'permalink', 'domain', 'url', 'selftext', 'title', 'score']
After cleaning: (361056, 13)
Saved:
 train: 288844
 val: 36106
 test: 36106
Example instruction:
 Write a joke.
LEVEL: easy
SETTING: online
TONE: dry
Example response:
 What do the Pope and a Christmas Tree have in common?
Their balls are just for decoration. ...


In [ ]:

import os, json, random
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

TEST_PATH = "./splits/test.jsonl"
OUT_PATH  = "./outputs/baseline_samples/distilgpt2_baseline_3.jsonl"

MODEL_NAME = "distilgpt2"
N_SAMPLES = 100          
MAX_NEW_TOKENS = 80

def load_examples(path, n):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    random.shuffle(rows)
    return rows[:n]

@torch.no_grad()
def main():
    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
    model.eval()

   
    tok.pad_token = tok.eos_token

    samples = load_examples(TEST_PATH, N_SAMPLES)

    with open(OUT_PATH, "w", encoding="utf-8") as out:
        for i, ex in enumerate(samples, 1):
            
            prompt = ex["instruction"].strip() + "\nJoke:\n"

            inputs = tok(prompt, return_tensors="pt").to(device)

            gen_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.9,
                top_p=0.95,
                repetition_penalty=1.1,
                pad_token_id=tok.eos_token_id,
            )

            full_text = tok.decode(gen_ids[0], skip_special_tokens=True)
            prediction = full_text[len(prompt):].strip()

            out.write(json.dumps({
                "instruction": ex["instruction"],
                "reference": ex["response"],
                "prediction": prediction
            }, ensure_ascii=False) + "\n")

            if i % 20 == 0:
                print(f"Generated {i}/{N_SAMPLES}")

    print(" Saved baseline to:", OUT_PATH)

if __name__ == "__main__":
    main()


Device: cuda


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generated 20/100
Generated 40/100
Generated 60/100
Generated 80/100
Generated 100/100
✅ Saved baseline to: ./outputs/baseline_samples/distilgpt2_baseline_3.jsonl


In [ ]:

import os, json, time, math
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "distilgpt2"
TRAIN_PATH = "./splits/train.jsonl"
VAL_PATH   = "./splits/val.jsonl"
OUT_DIR    = "./checkpoints/try3/distilgpt2_lora_finetuned3"

EPOCHS = 2
BATCH_SIZE = 8
MAX_LEN = 256
LR = 2e-4
WARMUP_RATIO = 0.05
GRAD_CLIP = 1.0
LOG_EVERY = 100
GRAD_ACCUM_STEPS = 1  

PROMPT_TMPL = "{inst}\nJoke:\n"
FULL_TMPL   = "{inst}\nJoke:\n{resp}"

os.makedirs(OUT_DIR, exist_ok=True)

class JsonlDataset(Dataset):
    def __init__(self, path):
        self.rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                self.rows.append(json.loads(line))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]

def collate_fn(batch, tok, max_len):
    input_ids_list, attn_list, labels_list = [], [], []

    for ex in batch:
        inst = ex["instruction"].strip()
        resp = ex["response"].strip()

        prompt = PROMPT_TMPL.format(inst=inst)
        full   = FULL_TMPL.format(inst=inst, resp=resp)

        enc_full = tok(full, truncation=True, max_length=max_len, add_special_tokens=False)
        enc_prompt = tok(prompt, truncation=True, max_length=max_len, add_special_tokens=False)

        input_ids = enc_full["input_ids"]
        attn = enc_full["attention_mask"]

        labels = input_ids.copy()
        prompt_len = len(enc_prompt["input_ids"])
        labels[:prompt_len] = [-100] * prompt_len

        input_ids_list.append(torch.tensor(input_ids, dtype=torch.long))
        attn_list.append(torch.tensor(attn, dtype=torch.long))
        labels_list.append(torch.tensor(labels, dtype=torch.long))

    input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids_list, batch_first=True, padding_value=tok.pad_token_id
    )
    attn_mask = torch.nn.utils.rnn.pad_sequence(
        attn_list, batch_first=True, padding_value=0
    )
    labels = torch.nn.utils.rnn.pad_sequence(
        labels_list, batch_first=True, padding_value=-100
    )

    return {"input_ids": input_ids, "attention_mask": attn_mask, "labels": labels}

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    n = 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        total_loss += out.loss.item()
        n += 1
    return total_loss / max(n, 1)

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    tok.pad_token = tok.eos_token

    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    base.resize_token_embeddings(len(tok))

    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["c_attn", "c_proj"],
    )
    model = get_peft_model(base, lora_cfg).to(device)
    model.print_trainable_parameters()

    train_ds = JsonlDataset(TRAIN_PATH)
    val_ds   = JsonlDataset(VAL_PATH)

    pin = (device == "cuda")
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=pin,
        collate_fn=lambda b: collate_fn(b, tok, MAX_LEN),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=pin,
        collate_fn=lambda b: collate_fn(b, tok, MAX_LEN),
    )

    optim = torch.optim.AdamW(model.parameters(), lr=LR)

    opt_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
    total_steps = max(1, EPOCHS * opt_steps_per_epoch)
    warmup_steps = int(WARMUP_RATIO * total_steps)

    sched = get_linear_schedule_with_warmup(
        optim,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    use_amp = (device == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_val = float("inf")
    step_count = 0

    with open(os.path.join(OUT_DIR, "run_config.json"), "w", encoding="utf-8") as f:
        json.dump({
            "model_name": MODEL_NAME,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "max_len": MAX_LEN,
            "lr": LR,
            "warmup_ratio": WARMUP_RATIO,
            "grad_accum_steps": GRAD_ACCUM_STEPS,
            "prompt_template": "instruction + \\nJoke:\\n + response"
        }, f, indent=2)

    for epoch in range(EPOCHS):
        model.train()
        t0 = time.time()
        running = 0.0

        optim.zero_grad(set_to_none=True)

        for step, batch in enumerate(train_loader, 1):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

            with torch.cuda.amp.autocast(enabled=use_amp):
                out = model(**batch)
                loss = out.loss / GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()
            running += loss.item()

            if step % GRAD_ACCUM_STEPS == 0:
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

                scaler.step(optim)
                scaler.update()
                optim.zero_grad(set_to_none=True)
                sched.step()

                step_count += 1

                if step_count % LOG_EVERY == 0:
                    lr_now = sched.get_last_lr()[0]
                    avg_loss = running / (LOG_EVERY * GRAD_ACCUM_STEPS)
                    print(f"epoch {epoch} opt_step {step_count} loss {avg_loss:.4f} lr {lr_now:.2e}")
                    running = 0.0

        
        if len(train_loader) % GRAD_ACCUM_STEPS != 0:
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optim)
            scaler.update()
            optim.zero_grad(set_to_none=True)
            sched.step()
            step_count += 1

        val_loss = evaluate(model, val_loader, device)
        dt = time.time() - t0
        print(f"epoch {epoch} val_loss {val_loss:.4f} time {dt/60:.1f} min")

        if val_loss < best_val:
            best_val = val_loss
            model.save_pretrained(OUT_DIR)
            tok.save_pretrained(OUT_DIR)
            print("Saved best checkpoint to:", OUT_DIR)

    print("Done. Best val_loss:", best_val)

if __name__ == "__main__":
    main()


Device: cuda


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 405,504 || all params: 82,318,080 || trainable%: 0.4926


C:\Users\hamoud\AppData\Local\Temp\ipykernel_27236\1738968663.py:147: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\hamoud\AppData\Local\Temp\ipykernel_27236\1738968663.py:174: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


epoch 0 opt_step 100 loss 4.4411 lr 5.54e-06
epoch 0 opt_step 200 loss 4.4430 lr 1.11e-05
epoch 0 opt_step 300 loss 4.2833 lr 1.66e-05
epoch 0 opt_step 400 loss 4.1701 lr 2.22e-05
epoch 0 opt_step 500 loss 4.0549 lr 2.77e-05
epoch 0 opt_step 600 loss 4.0963 lr 3.32e-05
epoch 0 opt_step 700 loss 4.0503 lr 3.88e-05
epoch 0 opt_step 800 loss 4.0271 lr 4.43e-05
epoch 0 opt_step 900 loss 4.0176 lr 4.99e-05
epoch 0 opt_step 1000 loss 3.9968 lr 5.54e-05
epoch 0 opt_step 1100 loss 4.0079 lr 6.09e-05
epoch 0 opt_step 1200 loss 3.9818 lr 6.65e-05
epoch 0 opt_step 1300 loss 3.9919 lr 7.20e-05
epoch 0 opt_step 1400 loss 3.9377 lr 7.76e-05
epoch 0 opt_step 1500 loss 3.9553 lr 8.31e-05
epoch 0 opt_step 1600 loss 3.9485 lr 8.86e-05
epoch 0 opt_step 1700 loss 3.9284 lr 9.42e-05
epoch 0 opt_step 1800 loss 3.8884 lr 9.97e-05
epoch 0 opt_step 1900 loss 3.9054 lr 1.05e-04
epoch 0 opt_step 2000 loss 3.9283 lr 1.11e-04
epoch 0 opt_step 2100 loss 3.8979 lr 1.16e-04
epoch 0 opt_step 2200 loss 3.9538 lr 1.22e-

In [ ]:
import os, json, random
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

CKPT_DIR  = "./checkpoints/try3/distilgpt2_lora_finetuned3"
TEST_PATH = "./splits/test.jsonl"
OUT_PATH  = "./outputs/finetuned_samples/distilgpt2_lora_finetuned3.jsonl"

N_SAMPLES = 100
MAX_NEW_TOKENS = 80

def load_examples(path, n):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    random.shuffle(rows)
    return rows[:n]

@torch.no_grad()
def main():
    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    tok = AutoTokenizer.from_pretrained(CKPT_DIR)
    tok.pad_token = tok.eos_token

    base = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)
    model = PeftModel.from_pretrained(base, CKPT_DIR).to(device)
    model.eval()

    samples = load_examples(TEST_PATH, N_SAMPLES)

    with open(OUT_PATH, "w", encoding="utf-8") as out:
        for i, ex in enumerate(samples, 1):
            prompt = ex["instruction"].strip() + "\nJoke:\n"
            inputs = tok(prompt, return_tensors="pt").to(device)

            gen_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.8,
                top_p=0.95,
                repetition_penalty=1.1,
                no_repeat_ngram_size=3,   
                pad_token_id=tok.eos_token_id,
            )

            full_text = tok.decode(gen_ids[0], skip_special_tokens=True)
            prediction = full_text[len(prompt):].strip()

            out.write(json.dumps({
                "instruction": ex["instruction"],
                "reference": ex["response"],
                "prediction": prediction
            }, ensure_ascii=False) + "\n")

            if i % 20 == 0:
                print(f"Generated {i}/{N_SAMPLES}")

    print(" Saved finetuned generations to:", OUT_PATH)

if __name__ == "__main__":
    main()


Device: cuda


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generated 20/100
Generated 40/100
Generated 60/100
Generated 80/100
Generated 100/100
✅ Saved finetuned generations to: ./outputs/finetuned_samples/distilgpt2_lora_finetuned3.jsonl


In [1]:


import os
import random
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel


try:
    LEVEL_OPTIONS = LEVELS  
except NameError:
    LEVEL_OPTIONS = ["easy", "medium", "hard"]

try:
    SETTING_OPTIONS = SETTINGS
except NameError:
    SETTING_OPTIONS = ["office", "school", "family dinner", "online", "bar"]

try:
    TONE_OPTIONS = TONES
except NameError:
    TONE_OPTIONS = ["sarcastic", "dry", "wholesome", "absurd"]

try:
    make_instruction_fn = make_instruction
except NameError:
    def make_instruction_fn(level: str, setting: str, tone: str) -> str:
        return (
            "Write a joke.\n"
            f"LEVEL: {level}\n"
            f"SETTING: {setting}\n"
            f"TONE: {tone}"
        )

CKPT_DIR = "./checkpoints/try3/distilgpt2_lora_finetuned3"
BASE_MODEL_NAME = "distilgpt2"

_device = "cuda" if torch.cuda.is_available() else "cpu"
_tok = None
_model = None


def _load_finetuned_model():
    
    global _tok, _model
    if _model is not None and _tok is not None:
        return

    if not os.path.isdir(CKPT_DIR):
        raise RuntimeError(
            f"Checkpoint directory not found: {CKPT_DIR}.\n"
            "Make sure you have already trained and saved the LoRA model."
        )

    _tok = AutoTokenizer.from_pretrained(CKPT_DIR)
    _tok.pad_token = _tok.eos_token

    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME).to(_device)
    _model = PeftModel.from_pretrained(base, CKPT_DIR).to(_device)
    _model.eval()


def _build_instruction(level: str, setting: str, tone: str, extra: str) -> str:
    inst = make_instruction_fn(level, setting, tone)
    extra = extra.strip()
    if extra:
        inst += "\n\nExtra details:\n" + extra
    return inst




header = widgets.HTML(
    """
    <div style="background: linear-gradient(135deg, #0f172a, #1e293b); padding: 18px 20px; border-radius: 12px; color: white;">
        <h2 style="margin: 0; font-weight: 600; font-size: 1.35rem;">Joke Generator Studio</h2>
        <p style="margin: 4px 0 0; font-size: 0.9rem; opacity: 0.85;">
            Reddit Jokes Generator: Team: BOUCHENTOUF Omar, MOHAMED JOURFI, ABOUBEKRIN MOHAMED SALEM, Stephie JEAN JOSEPH
        </p>
    </div>
    """
)

level_dd = widgets.ToggleButtons(
    options=LEVEL_OPTIONS,
    description="Level:",
    button_style="",
    style={"button_width": "90px"},
)

setting_dd = widgets.Dropdown(
    options=SETTING_OPTIONS,
    description="Setting:",
)

tone_dd = widgets.ToggleButtons(
    options=TONE_OPTIONS,
    description="Tone:",
    button_style="",
    style={"button_width": "110px"},
)

extra_text = widgets.Textarea(
    description="Extra cue:",
    placeholder="Optional: add specific topic, characters, or constraints...",
    layout=widgets.Layout(width="100%", height="80px"),
)

max_tokens_slider = widgets.IntSlider(
    value=80,
    min=20,
    max=200,
    step=10,
    description="Max tokens",
    continuous_update=False,
)

temperature_slider = widgets.FloatSlider(
    value=0.8,
    min=0.1,
    max=1.5,
    step=0.05,
    description="Temperature",
    continuous_update=False,
)

top_p_slider = widgets.FloatSlider(
    value=0.95,
    min=0.5,
    max=1.0,
    step=0.01,
    description="Top-p",
    continuous_update=False,
)

generate_btn = widgets.Button(
    description="Generate joke",
    button_style="primary",
    tooltip="Click to sample from the finetuned model",
    layout=widgets.Layout(width="160px", height="40px"),
)

output = widgets.Output(layout=widgets.Layout(border="1px solid #e5e7eb", padding="12px", border_radius="10px", background_color="#ffffff"))


def on_generate_clicked(_):
    with output:
        clear_output()
        print(f"Device: {_device}")
        try:
            _load_finetuned_model()
        except Exception as e:
            print("Could not load finetuned model.")
            print(str(e))
            return

        lvl = level_dd.value
        setting = setting_dd.value
        tone = tone_dd.value
        extra = extra_text.value

        instruction = _build_instruction(lvl, setting, tone, extra)
        prompt = instruction + "\nJoke:\n"

        inputs = _tok(prompt, return_tensors="pt").to(_device)
        gen_ids = _model.generate(
            **inputs,
            max_new_tokens=max_tokens_slider.value,
            do_sample=True,
            temperature=float(temperature_slider.value),
            top_p=float(top_p_slider.value),
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            pad_token_id=_tok.eos_token_id,
        )

        full_text = _tok.decode(gen_ids[0], skip_special_tokens=True)
        joke = full_text[len(prompt):].strip()

        print("Instruction:")
        print(instruction)
        print("\nGenerated joke:\n")
        print(joke)


generate_btn.on_click(on_generate_clicked)

controls_top = widgets.HBox([
    widgets.VBox([level_dd]),
    widgets.VBox([setting_dd]),
    widgets.VBox([tone_dd]),
])

sampling_row = widgets.HBox([
    max_tokens_slider,
    temperature_slider,
    top_p_slider,
])

button_row = widgets.HBox([
    generate_btn,
], layout=widgets.Layout(justify_content="flex-start", padding="6px 0"))

card = widgets.VBox(
    [
        header,
        widgets.HTML("<h3 style='margin: 14px 0 6px;'>Controls</h3>"),
        controls_top,
        widgets.HTML("<h3 style='margin: 14px 0 6px;'>Extra guidance</h3>"),
        extra_text,
        widgets.HTML("<h3 style='margin: 14px 0 6px;'>Sampling</h3>"),
        sampling_row,
        button_row,
        widgets.HTML("<h3 style='margin: 14px 0 6px;'>Output</h3>"),
        output,
    ],
    layout=widgets.Layout(
        border="1px solid #d1d5db",
        border_radius="14px",
        padding="14px",
        background_color="#f9fafb",
        width="100%",
    ),
)

display(card)

In [2]:
import os
import torch
import streamlit as st
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# -----------------------------
# Settings
# -----------------------------
LEVEL_OPTIONS = ["easy", "medium", "hard"]
SETTING_OPTIONS = ["office", "school", "family dinner", "online", "bar"]
TONE_OPTIONS = ["sarcastic", "dry", "wholesome", "absurd"]

CKPT_DIR = "./checkpoints/try3/distilgpt2_lora_finetuned3"
BASE_MODEL_NAME = "distilgpt2"
_device = "cuda" if torch.cuda.is_available() else "cpu"

_tok = None
_model = None

# -----------------------------
# Load model
# -----------------------------
def load_model():
    global _tok, _model
    if _model is not None and _tok is not None:
        return

    if not os.path.isdir(CKPT_DIR):
        st.error(f"Checkpoint directory not found: {CKPT_DIR}")
        return

    _tok = AutoTokenizer.from_pretrained(CKPT_DIR)
    _tok.pad_token = _tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME).to(_device)
    _model = PeftModel.from_pretrained(base, CKPT_DIR).to(_device)
    _model.eval()

# -----------------------------
# Instruction builder
# -----------------------------
def make_instruction(level, setting, tone):
    return f"Write a joke.\nLEVEL: {level}\nSETTING: {setting}\nTONE: {tone}"

def build_prompt(level, setting, tone, extra):
    inst = make_instruction(level, setting, tone)
    extra = extra.strip()
    if extra:
        inst += "\n\nExtra details:\n" + extra
    return inst + "\nJoke:\n"

# -----------------------------
# Streamlit UI
# -----------------------------
st.set_page_config(page_title="Reddit Joke Generator", layout="wide")

st.markdown("""
# 🎭 Reddit Jokes Generator
Team: BOUCHENTOUF Omar, MOHAMED JOURFI, ABOUBEKRIN MOHAMED SALEM, Stephie JEAN JOSEPH
""")

# Sidebar controls
st.sidebar.header("Controls")
level = st.sidebar.radio("Level", LEVEL_OPTIONS)
setting = st.sidebar.selectbox("Setting", SETTING_OPTIONS)
tone = st.sidebar.radio("Tone", TONE_OPTIONS)
extra = st.sidebar.text_area("Extra guidance", placeholder="Optional: topic, characters, constraints...")

st.sidebar.header("Sampling")
max_tokens = st.sidebar.slider("Max tokens", 20, 200, 80, step=10)
temperature = st.sidebar.slider("Temperature", 0.1, 1.5, 0.8, step=0.05)
top_p = st.sidebar.slider("Top-p", 0.5, 1.0, 0.95, step=0.01)

# Generate button
if st.button("Generate joke"):
    load_model()
    if _model is not None:
        prompt = build_prompt(level, setting, tone, extra)
        inputs = _tok(prompt, return_tensors="pt").to(_device)
        gen_ids = _model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=float(temperature),
            top_p=float(top_p),
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            pad_token_id=_tok.eos_token_id,
        )
        full_text = _tok.decode(gen_ids[0], skip_special_tokens=True)
        joke = full_text[len(prompt):].strip()

        st.markdown("### Instruction")
        st.text(prompt)
        st.markdown("### Generated Joke")
        st.success(joke)

2026-03-03 22:24:17.905 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-03 22:24:17.906 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-03 22:24:17.906 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-03 22:24:17.907 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-03 22:24:17.908 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-03 22:24:17.909 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-03 22:24:17.909 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-03 22:24:17.910 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar